In [1]:
import sys
sys.path.insert(0, "_src")
from financial_tools import FMPClient

import pandas as pd
import numpy as np
from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)
# --- Configuration ---

# --- Instantiate the Classes ---
client1 = FMPClient()
filings_8k = client1.get_data('sec-filings-8k', "ALL")

https://financialmodelingprep.com/stable/sec-filings-8k?&from=2026-04-23&to=2026-04-24&page=0&limit=300&apikey=5nTvG2lrPgVl9QBo7AadhieWKbjUQtUm
   Fetching sec-filings-8k for ALL...


In [4]:
filings_8k[17]['link']


'https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/0001437749-26-013337-index.htm'

In [5]:
filings_8k[17]['finalLink']

'https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm'

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import requests
from pathlib import Path
import time


SEC_BASE = "https://www.sec.gov"

session = requests.Session()
session.headers.update({
    "User-Agent": "Dave Zhuo zhuo.longhao@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})


def download_sec_html(url, timeout=(5, 20)):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text


def html_to_clean_text(html):
    soup = BeautifulSoup(html, "html.parser")

    # remove junk
    for tag in soup(["script", "style", "noscript", "nav", "header", "footer"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    return text


def extract_sec_documents(filing_detail_url, filing_detail_html):
    """
    Parse the SEC filing detail page and return document links.
    """
    soup = BeautifulSoup(filing_detail_html, "html.parser")

    docs = []

    table = soup.find("table", class_="tableFile", summary="Document Format Files")
    if not table:
        return docs

    rows = table.find_all("tr")[1:]

    for row in rows:
        cells = row.find_all("td")
        if len(cells) < 5:
            continue

        seq = cells[0].get_text(" ", strip=True)
        description = cells[1].get_text(" ", strip=True)
        document_cell = cells[2]
        doc_type = cells[3].get_text(" ", strip=True)
        size = cells[4].get_text(" ", strip=True)

        a = document_cell.find("a")
        if not a:
            continue

        href = a.get("href")
        document_name = a.get_text(" ", strip=True)

        # Important:
        # /ix?doc=/Archives/... is the inline XBRL viewer.
        # We want the real document path after doc=
        if href.startswith("/ix?doc="):
            href = href.replace("/ix?doc=", "")

        full_url = urljoin(SEC_BASE, href)

        docs.append({
            "seq": seq,
            "description": description,
            "document": document_name,
            "type": doc_type,
            "size": size,
            "url": full_url,
        })

    return docs

def has_keywords_in_text(text):
    text = text.lower()

    keywords = [
        "credit agreement",
        "amended and restated credit agreement",
        "loan agreement",
        "revolving credit facility",
        "term loan",
        "entry into a material definitive agreement",
        "item 1.01",
    ]

    return any(keyword in text for keyword in keywords)

def search_actual_sec_documents(filing_detail_url):
    index_html = download_sec_html(filing_detail_url)
    docs = extract_sec_documents(filing_detail_url, index_html)

    results = []

    for doc in docs:
        doc_type = doc["type"].upper()

        # Usually useful documents only
        if doc_type not in ["8-K", "EX-10.1", "EX-10.2", "EX-99.1"]:
            continue

        try:
            html = download_sec_html(doc["url"])
            text = html_to_clean_text(html)
            matched = has_keywords_in_text(text)

            results.append({
                "type": doc["type"],
                "description": doc["description"],
                "document": doc["document"],
                "url": doc["url"],
                "matched": matched,
                "text": text,
            })

            time.sleep(0.2)

        except Exception as e:
            results.append({
                "type": doc["type"],
                "description": doc["description"],
                "document": doc["document"],
                "url": doc["url"],
                "matched": False,
                "text": "",
                "error": str(e),
            })

    return results

all_results = []

for i, filing in enumerate(filings_8k, start=1):
    symbol = filing.get("symbol")
    filing_date = filing.get("filingDate")
    filing_detail_url = filing.get("finalLink")

    print(f"[{i}/{len(filings_8k)}] Assessing: {symbol} | {filing_date}")

    try:
        doc_results = search_actual_sec_documents(filing_detail_url)

        matched_docs = [r for r in doc_results if r["matched"]]

        for r in doc_results:
            all_results.append({
                "symbol": symbol,
                "filingDate": filing_date,
                "filing_detail_url": filing_detail_url,
                "document_type": r.get("type"),
                "document_description": r.get("description"),
                "document_url": r.get("url"),
                "matched": r.get("matched"),
                "error": r.get("error"),
            })

        if matched_docs:
            print(f"  MATCH: {len(matched_docs)} document(s)")
        else:
            print("  No match")

    except Exception as e:
        print(f"  ERROR: {e}")

        all_results.append({
            "symbol": symbol,
            "filingDate": filing_date,
            "filing_detail_url": filing_detail_url,
            "document_type": None,
            "document_description": None,
            "document_url": None,
            "matched": False,
            "error": str(e),
        })

    time.sleep(0.2)

In [11]:
filing_detail_url = filings_8k[17]['link']
index_html = download_sec_html(filing_detail_url)

docs = extract_sec_documents(filing_detail_url, index_html)

for doc in docs:
    print(doc["type"], doc["description"], doc["url"])

8-K FORM 8-K https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
EX-10.1 EXHIBIT 10.1 https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm
 Complete submission text file https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/0001437749-26-013337.txt


In [14]:
for doc in docs:
    print(doc["url"])
    results = search_actual_sec_documents(
        filing_detail_url
    )

    for r in results:
        print(r["type"], r["description"], r["matched"], r["url"])

https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
8-K FORM 8-K True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
EX-10.1 EXHIBIT 10.1 True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm
https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm
8-K FORM 8-K True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
EX-10.1 EXHIBIT 10.1 True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm
https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/0001437749-26-013337.txt
8-K FORM 8-K True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
EX-10.1 EXHIBIT 10.1 True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm


In [ ]:
def has_keywords_in_text(text):
    text = text.lower()

    keywords = [
        "credit agreement",
        "amended and restated credit agreement",
        "loan agreement",
        "revolving credit facility",
        "term loan",
        "entry into a material definitive agreement",
        "item 1.01",
    ]

    return any(keyword in text for keyword in keywords)




In [42]:
results = search_actual_sec_documents(
    "https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/0001437749-26-013337-index.htm"
)

for r in results:
    print(r["type"], r["description"], r["matched"], r["url"])

8-K FORM 8-K True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/pdfs20260423_8k.htm
EX-10.1 EXHIBIT 10.1 True https://www.sec.gov/Archives/edgar/data/1120914/000143774926013337/ex_950183.htm


[1/20] Assessing: SUNE | 2024-03-04 00:00:00
  No match
[2/20] Assessing: CGBDL | 2024-03-04 00:00:00
  No match
[3/20] Assessing: SLE | 2024-03-01 00:00:00
  No match
[4/20] Assessing: SBC | 2024-03-01 00:00:00
  No match
[5/20] Assessing: SBCWW | 2024-03-01 00:00:00
  No match
[6/20] Assessing: TSNU | 2024-03-01 00:00:00
  No match
[7/20] Assessing: SHOT | 2024-03-01 00:00:00
  No match
[8/20] Assessing: SHOTW | 2024-03-01 00:00:00
  No match
[9/20] Assessing: EMPD | 2024-03-01 00:00:00
  No match
[10/20] Assessing: FTIVU | 2024-03-01 00:00:00
  No match
[11/20] Assessing: FTIV | 2024-03-01 00:00:00
  No match
[12/20] Assessing: PWPPW | 2024-03-01 00:00:00
  No match
[13/20] Assessing: LUXHP | 2024-03-01 00:00:00
  No match
[14/20] Assessing: RIME | 2024-03-01 00:00:00
  No match
[15/20] Assessing: NTRP | 2024-03-01 00:00:00
  No match
[16/20] Assessing: BJDX | 2024-03-04 00:00:00
  No match
[17/20] Assessing: CLNV | 2024-03-04 00:00:00
  No match
[18/20] Assessing: FFAIW | 2024-03-0